# 03 · SQLAlchemy 2.0 & Alembic — Python ⇄ PostgreSQL, and safe schema changes

**Time:** ~60 min · **Needs:** PostgreSQL running

> **Mental model.**
> * **SQLAlchemy** translates between Python objects and SQL rows (an *ORM*). Your classes describe tables.
> * A **Session** is a *shopping basket* + *notepad*: it remembers the objects you touched and writes
>   them to the database in one go (`commit`).
> * **Alembic** is *git for your database schema*: every change is a small, ordered, reversible script (a *migration*).

**Contents:** models · sessions · queries · relationships · the N+1 problem · transactions · bulk updates ·
reading the real TeamDesk models · running real Alembic migrations on a scratch database

# SQLAlchemy Imports Reference

## 1. Core SQLAlchemy Components

| Import | Purpose |
|---|---|
| `Column` | Defines a column in a database table (legacy ORM style). |
| `ForeignKey` | Defines a foreign key relationship between tables. |
| `MetaData` | Stores metadata about database tables and schemas. |
| `String` | Defines a string data type for a column. |
| `Table` | Defines a database table explicitly using Core API. |
| `create_engine` | Creates a database engine and manages database connections. |
| `event` | Registers event listeners for database operations. |
| `func` | Provides SQL functions such as `COUNT()`, `AVG()`, and `MAX()`. |
| `select` | Constructs SQL SELECT statements. |
| `text` | Executes raw SQL expressions. |
| `update` | Constructs SQL UPDATE statements. |

## 2. SQLAlchemy ORM Components

| Import | Purpose |
|---|---|
| `DeclarativeBase` | Base class for defining ORM models. |
| `Mapped` | Provides Python type annotations for mapped attributes. |
| `Session` | Manages database interactions and transactions. |
| `relationship` | Defines relationships between ORM models. |

## 3. Relationship Loading Strategies

These control how SQLAlchemy loads related objects from the database.

| Import | Purpose |
|---|---|
| `joinedload` | Loads related data using SQL JOINs. |
| `selectinload` | Loads related data using a separate SELECT query with an IN clause. |

In [1]:
import sys, os 
from pathlib import Path
from contextlib import contextmanager

ROOT = Path.cwd().parent  # notebooks/ -> repo root
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # settings read .env relative to cwd

from sqlalchemy import (Column, ForeignKey, MetaData, String, Table, create_engine, event, func, select, text, update) 
from sqlalchemy.orm import (DeclarativeBase, Mapped, Session, joinedload, relationship, selectinload)

from backend.app.core.config import get_settings

URL = get_settings().database_url
engine = create_engine(URL)
